# Python For Machine Learning 6: Preprocessing
In the past sessions we were using Penguins dataset. The penguins dataset is misleading in a way, **real datasets are never that tidy.** They have gaps, text columns a model can't read, and extreme values that distort everything.

Before we move onto more complicated topics on machine learning, we will learn more about **preprocessing**. Preprocessing turns a raw, messy table into a clean numeric table a model can actually learn from. We'll use the **Titanic** dataset (who survived the 1912 shipwreck), which consists missing ages, text columns, and wild ticket fares.

By the end you should be able to take a raw CSV and produce a clean feature matrix ready for the Session 5 workflow.


## The preprocessing checklist

Preprocessing follows a series of steps:

1. **Inspect** the data : what's there, what's missing, what's text vs numbers
2. **Handle missing values** : drop them or fill them in
3. **Encode categorical features** : turn text into numbers (one-hot encoding)
4. **Detect & handle outliers** : extreme values that can distort the model
5. **Scale features** : put numbers on a comparable range (for some models)

The output of all this is a clean `X` (and `y`) you can drop straight into `train_test_split` → `fit` → `score`.

## 1. Load and inspect

Before you preprocess the dataset, you need to understand the dataset. The following pandas functions are used for the purpose of inspecting the dataset and understanding it as we have discussed in the classes before.

The 3 inspection tools you already know from Session 4 are crucial here:
- `df.info()`: column types and how many non-null values each has
- `df.isnull().sum()`: exactly how many missing values per column
- `df.describe()`: ranges, means, and a first hint of outliers (look at min vs max)


In [43]:
import pandas as pd
import numpy as np

# Load Titanic and keep a classic subset of columns
full = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")
# We are just working with a subset of titanic dataset here
df_not_preprocessed = full[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]].copy()
df = df_not_preprocessed.copy()

print("Shape (rows, columns):", df.shape)
df.head()

Shape (rows, columns): (891, 8)


,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [35]:
# Where are the holes, and what types are we dealing with?
print("Missing values per column:")
print(df.isnull().sum())

print("\nColumn types:")
print(df.dtypes)

Missing values per column:
survived      0
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
dtype: int64

Column types:
survived      int64
pclass        int64
sex          object
age         float64
sibsp         int64
parch         int64
fare        float64
embarked     object
dtype: object


In [27]:
# Numeric summary — watch the min/max for signs of outliers
df.describe()


,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


Read what just came back:

- **`age` has 177 missing values** (about 20% of the data) and **`embarked` has 2**. We must deal with these, models reject `NaN`.
- **`sex` and `embarked` are text** (`object`), a model can't do arithmetic on the word "male." These need encoding.
- **`fare` runs from 0 to over 500** with a mean near 32, that huge max is a sign of outliers we'll investigate.

### a. Handle missing values

A model cannot train on `NaN`. You have three options, and choosing well matters:

- **Drop the rows** (`dropna`) : what we did in Session 5. Fine when only a handful are missing, but here it would throw away ~20% of the data along with the 177 ages. Wasteful.
- **Drop the column** : sensible only if a column is *mostly* empty and not very useful.
- **Impute** (fill in) the gaps : replace each missing value with a sensible substitute. For **numeric** columns we usually use the **median** (robust to skew/outliers, better than the mean for skewed data like age/fare). For **categorical** columns we use the **mode** (the most frequent category).

We'll impute: median for `age`, mode for `embarked`.

```python
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
```

<br>

![Mean, Median and Mode](https://raw.githubusercontent.com/aksho-sh/CollabImages/refs/heads/main/MeanMedianMode.png)
_fig: Using mean, median and mode_

<br>

![Handling Missing values](https://raw.githubusercontent.com/aksho-sh/CollabImages/refs/heads/main/MissingValues.png)
_fig: Using mean, median and mode_

In [36]:
# Impute age with the median (robust to the skew in ages)
df["age"] = df["age"].fillna(df["age"].median())

# Impute the 2 missing embarked values with the most common port (the mode)
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])

print("Missing values now:")
print(df.isnull().sum())

Missing values now:
survived    0
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
dtype: int64


In [37]:
df

,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S
887,1,1,female,19.0,0,0,30.0000,S
888,0,3,female,28.0,1,2,23.4500,S
889,1,1,male,26.0,0,0,30.0000,C


### Task 1

- Re-run the load cell (the one where you loaded the dataset into the dataframe using a link) to get the missing values back, then try filling `age` with the **mean** instead of the median (`df["age"].mean()`). Print both values. How far apart are the mean and median? (A big gap is a sign the column is skewed, which is exactly when the median is the safer choice.)

In [38]:
# Task 1 — compare mean vs median for age
print("Mean age:", df["age"].mean())
print("Median age:", df["age"].median())

Mean age: 29.36158249158249
Median age: 28.0


## 3. Encode categorical features

`sex` and `embarked` are text. Models work only with numbers, so we must convert them. Two common ways:

- **Label / ordinal encoding** : map each category to an integer (`male`→0, `female`→1). Only appropriate when the categories have a real *order* (small/medium/large), or for tree-based models.
- **One-hot encoding** : create a separate 0/1 column for each category. This is the safe default for unordered (nominal) categories. pandas does it with `get_dummies`.

<br>

![OneHotEncoding](https://raw.githubusercontent.com/aksho-sh/CollabImages/refs/heads/main/OneHotEncoding.png)
_fig: One-hot encoding to deal with categorical values_


```python
df = pd.get_dummies(df, columns=["sex", "embarked"], drop_first=True, dtype=int)
```

`drop_first=True` drops one category per feature (e.g. it keeps `sex_male` and drops `sex_female`), because "not male" already tells you "female", keeping both is redundant. `dtype=int` gives clean 0/1 values.

In [39]:
df = pd.get_dummies(df, columns=["sex", "embarked"], drop_first=True, dtype=int)

print("Columns after one-hot encoding:")
print(list(df.columns))
df.head()

Columns after one-hot encoding:
['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare', 'sex_male', 'embarked_Q', 'embarked_S']


,survived,pclass,age,sibsp,parch,fare,sex_male,embarked_Q,embarked_S
0,0,3,22.0,1,0,7.2500,1,0,1
1,1,1,38.0,1,0,71.2833,0,0,0
2,1,3,26.0,0,0,7.9250,0,0,1
3,1,1,35.0,1,0,53.1000,0,0,1
4,0,3,35.0,0,0,8.0500,1,0,1


Easy rule to remember
* `drop_first`=False → Keep all dummy columns
* `drop_first`=True → Remove one dummy column

Notice the text columns are gone, replaced by numeric 0/1 columns like `sex_male`, `embarked_Q`, and `embarked_S`. Every column is now a number, the model can read the whole table.

### Task 2

- Re-run with `drop_first=False` and compare the columns. You'll see *both* `sex_male` and `sex_female` appear. Why is keeping both unnecessary (and occasionally harmful for some models)?

In [44]:
# Task 2 — try drop_first=False and inspect the columns
df = pd.get_dummies(df, columns=["sex", "embarked"], drop_first=False, dtype=int)

print("Columns after one-hot encoding:")
print(list(df.columns))
df.head()

Columns after one-hot encoding:
['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare', 'sex_female', 'sex_male', 'embarked_C', 'embarked_Q', 'embarked_S']


,survived,pclass,age,sibsp,parch,fare,sex_female,sex_male,embarked_C,embarked_Q,embarked_S
0,0,3,22.0,1,0,7.2500,0,1,0,0,1
1,1,1,38.0,1,0,71.2833,1,0,1,0,0
2,1,3,26.0,0,0,7.9250,1,0,0,0,1
3,1,1,35.0,1,0,53.1000,1,0,0,0,1
4,0,3,35.0,0,0,8.0500,0,1,0,0,1


## 4. Detect and handle outliers

`fare` had a suspicious maximum of 512 while most fares were under 32. **Outliers**, values far outside the normal range can distort some models and inflate error metrics. A standard way to detect them is the **IQR rule**:

- **Q1** = 25th percentile, **Q3** = 75th percentile, **IQR** = Q3 − Q1 (the middle 50% of the data)
- Anything below `Q1 − 1.5·IQR` or above `Q3 + 1.5·IQR` is flagged as an outlier.

Once detected, you have choices: **remove** the rows, or **cap** them (clip extreme values to the boundary). Capping is gentler — it keeps the row but tames the extreme. We'll cap.

**A word of caution:** an outlier is not automatically an error. A £512 fare might be a real first-class suite. Don't delete outliers reflexively — investigate first, and prefer capping over deleting unless you're sure the value is wrong.

<br>

![](https://raw.githubusercontent.com/aksho-sh/CollabImages/refs/heads/main/IQR.png)
_fig: Interquartile Range for outlier detection_

In [45]:
# Detect fare outliers with the IQR rule
q1 = df["fare"].quantile(0.25)
q3 = df["fare"].quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr

n_outliers = ((df["fare"] < lower) | (df["fare"] > upper)).sum()
print(f"IQR bounds for fare: {lower:.2f} to {upper:.2f}")
print(f"Number of fare outliers: {n_outliers}")

IQR bounds for fare: -26.72 to 65.63
Number of fare outliers: 116


In [46]:
# Cap (clip) the fares to the IQR bounds instead of deleting the rows
df["fare"] = df["fare"].clip(lower=lower, upper=upper)

print("Fare after capping:")
print(df["fare"].describe().round(2))

Fare after capping:
count    891.00
mean      24.05
std       20.48
min        0.00
25%        7.91
50%       14.45
75%       31.00
max       65.63
Name: fare, dtype: float64


### Task 3

- Apply the same IQR detection to the `age` column. How many ages are flagged as outliers?
- Try **removing** fare outliers instead of capping (filter with `df = df[df["fare"] <= upper]`) and compare how many rows you lose. Which approach do you think is better here, and why?

In [ ]:
# Task 3 — outlier detection on age, and removal vs capping

Q1 = df["age"].quantile(0.25)
Q3 = df["age"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

age_outliers = df[(df["age"] < lower) | (df["age"] > upper)]

print("Number of age outliers:", len(age_outliers))

Number of age outliers: 11


In [48]:
Q1 = df["fare"].quantile(0.25)
Q3 = df["fare"].quantile(0.75)

IQR = Q3 - Q1

upper = Q3 + 1.5 * IQR

original_rows = len(df)

df_removed = df[df["fare"] <= upper]

rows_lost = original_rows - len(df_removed)

print("Original rows:", original_rows)
print("Rows after removing fare outliers:", len(df_removed))
print("Rows lost:", rows_lost)

Original rows: 891
Rows after removing fare outliers: 891
Rows lost: 0


In [50]:
df["fare"] = df["fare"].clip(upper=upper)
df

,survived,pclass,age,sibsp,parch,fare,sex_female,sex_male,embarked_C,embarked_Q,embarked_S
0,0,3,22.0,1,0,7.2500,0,1,0,0,1
1,1,1,38.0,1,0,65.6344,1,0,1,0,0
2,1,3,26.0,0,0,7.9250,1,0,0,0,1
3,1,1,35.0,1,0,53.1000,1,0,0,0,1
4,0,3,35.0,0,0,8.0500,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,27.0,0,0,13.0000,0,1,0,0,1
887,1,1,19.0,0,0,30.0000,1,0,0,0,1
888,0,3,NaN,1,2,23.4500,1,0,0,0,1
889,1,1,26.0,0,0,30.0000,0,1,1,0,0


### Which is better?

For the Titanic dataset, capping is usually better because:

* The dataset is not very large.
* High fares can be genuine values, not mistakes.
* Capping reduces the effect of extreme values while keeping all passengers in the dataset.

## 5. Scale the features

<br>

![Scaling numerical features](https://raw.githubusercontent.com/aksho-sh/CollabImages/refs/heads/main/Scaling.png)
_fig: Scaling the features_

Remember Session 5, where KNN scored poorly because `body_mass_g` (in the thousands) drowned out `bill_length_mm` (in the tens)? The same issue lurks here: `fare` and `age` live on very different scales. **Feature scaling** puts every numeric column on a comparable range so no single feature dominates by sheer magnitude.

Two common scalers:
- **StandardScaler** : rescales each column to mean 0, standard deviation 1.
- **MinMaxScaler** : squashes each column into the range 0 to 1.

```python
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df[["age", "fare"]] = scaler.fit_transform(df[["age", "fare"]])
```

**Two important caveats:**
- **Tree-based models** (decision trees, random forests) **don't need scaling** — they split on thresholds, so magnitude is irrelevant. Scaling matters for distance and gradient-based models (KNN, logistic/linear regression, neural nets).
- In a real project you **fit the scaler on the training data only**, *after* the train/test split, otherwise information from the test set leaks into training. We scale here for demonstration; the optional section shows the leak-proof way.

In [51]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[["age", "fare"]] = scaler.fit_transform(df[["age", "fare"]])

# Now age and fare are centered near 0 with a similar spread
df[["age", "fare"]].describe().round(2)


,age,fare
count,714.00,891.00
mean,0.00,0.00
std,1.00,1.00
min,-2.02,-1.17
25%,-0.66,-0.79
50%,-0.12,-0.47
75%,0.57,0.34
max,3.47,2.03


## 5. Put it all together

We've gone from a messy table to a fully numeric, gap-free, scaled one. The payoff: it now drops straight into the Session 5 workflow. Let's predict **survival** to prove the cleaned data works. (We'll use a decision tree, which doesn't care about scaling, the point is that the data is now *usable at all*.)


In [52]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Features = everything except the label; label = survived
X = df.drop(columns=["survived"])
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = DecisionTreeClassifier(max_depth=4, random_state=0)
model.fit(X_train, y_train)
print("Accuracy on unseen passengers:", round(model.score(X_test, y_test), 3))

Accuracy on unseen passengers: 0.799


From a raw CSV full of gaps and text to a working survival predictor, that's the whole point of preprocessing. The model step was trivial; the *preparation* is where most real ML work actually happens.

## Recap

| Problem | Tool | Typical choice |
|---|---|---|
| Missing numeric values | `fillna(...)` | median |
| Missing categorical values | `fillna(mode)` | most frequent |
| Text / categorical columns | `pd.get_dummies(...)` | one-hot, `drop_first=True` |
| Extreme values | IQR rule + `clip(...)` | cap rather than delete |
| Different scales | `StandardScaler` / `MinMaxScaler` | only for non-tree models |

The order matters less than the habit: **always inspect first, then clean, then model.**

More on Handling Texts

## Standardizing text before encoding

One-hot encoding treats **every distinct string as its own category**. If the same value is written inconsistently, different capitalization or stray spaces, it splits into several *fake* categories. `"dog"`, `"Dog"`, and `" dog "` each become a separate column even though they all mean **dog**, which inflates your feature count and scatters one real signal across columns that should be one.

A quick look at the problem:

```python
messy = pd.Series(["Dog", "dog", " dog ", "Cat", "cat"]) # Imagine these are unique values in a column
messy.nunique()          # five "categories" for two real animals
```

The fix is to standardize the text first, usually two cleanups:

- `.str.strip()` — remove leading/trailing spaces
- `.str.lower()` — put everything in the same case

```python
clean = messy.str.strip().str.lower()
clean.nunique()          # 2 — collapses to just dog and cat
```

So before one-hot encoding, apply the same cleanup to your text columns:

```python
for col in ["sex", "embarked"]:
    df[col] = df[col].str.strip().str.lower()
```

For messier cases — `"M"` vs `"Male"` vs `"male"`, or typos — strip-and-lower won't merge genuine synonyms. Map the known variants to a single label with `.replace(...)`:

```python
df["sex"] = df["sex"].replace({"m": "male", "f": "female"})
```

**Heads-up:** lowercasing `embarked` turns C/Q/S into c/q/s, so the one-hot columns become `embarked_q` and `embarked_s` — match that case anywhere you reference them later.

## More on Scaling

**If you've finished the tasks, read through these.**

### The professional way: Pipelines (and no data leakage)

Doing each step by hand is great for learning, but in practice you bundle preprocessing and the model into a single `Pipeline`, often with a `ColumnTransformer` that applies different steps to different columns. The big win: when you call `.fit()` it learns the imputation values and scaling **from the training data only**, automatically avoiding the leakage we warned about.

```python
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

numeric = ["age", "fare", "sibsp", "parch", "pclass"]
categorical = ["sex", "embarked"]

pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                       ("scale", StandardScaler())]), numeric),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                       ("onehot", OneHotEncoder(drop="first"))]), categorical),
])

model = Pipeline([("prep", pre), ("clf", DecisionTreeClassifier(max_depth=4))])
# model.fit(X_train, y_train) now preprocesses + trains in one leak-free step
```

### Other tools worth knowing
- **`SimpleImputer`** : sklearn's own imputer (above); plays nicely with pipelines.
- **`RobustScaler`** : scaling based on the median and IQR; better than StandardScaler when outliers remain.
- **Target / frequency encoding** : alternatives to one-hot when a categorical column has *many* categories (one-hot would create too many columns).
- **`MinMaxScaler`** : when you specifically need values in [0, 1] (e.g. some neural networks).

### The golden rule
Fit every "learned" preprocessing step (imputation values, scaling statistics, encoders) on the **training data only**, then apply to the test data. Anything you learn from the test set and feed back into training is **data leakage**, and it makes your scores look better than they really are.